<a href="https://colab.research.google.com/github/BahruzHuseynov/Portfolio/blob/main/Medium/TensorFlow/TF4_Custom_Loss_Layer_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import tensorflow as tf
from tensorflow import keras
from keras import layers, losses, optimizers, metrics, Sequential

### Data Loading and Preparation

In [1]:
!pip install -q kaggle

In [2]:
from google.colab import files
files.upload()

Saving kaggle (1).json to kaggle (1).json


{'kaggle (1).json': b'{"username":"hbahruz","key":"4f925bc35f1b4f8677fc6f8061cceb06"}'}

In [3]:
!kaggle datasets download -d akhiljethwa/forest-vs-desert

Dataset URL: https://www.kaggle.com/datasets/akhiljethwa/forest-vs-desert
License(s): CC-BY-NC-SA-4.0
 93% 7.00M/7.54M [00:00<00:00, 71.8MB/s]
100% 7.54M/7.54M [00:00<00:00, 73.2MB/s]


In [4]:
import zipfile

dzip = 'forest-vs-desert.zip'

def extract_zip(file_path, extract_to='.'):
    with zipfile.ZipFile(file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

extract_zip(dzip, 'forest_vs_desert')

In [7]:
img_height, img_width = 64, 64

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'forest_vs_desert/Data',
    validation_split=0.2,  # Use 20% of training data for validation
    subset="training",
    seed=123,
    image_size=(img_height, img_width)
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'forest_vs_desert/Data',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width)
)

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=1024)
val_ds = val_ds.cache().prefetch(buffer_size=1024)

Found 802 files belonging to 2 classes.
Using 642 files for training.
Found 802 files belonging to 2 classes.
Using 160 files for validation.


### Custom Loss

In [22]:
from tensorflow.keras.losses import Loss

class MyBinaryCELoss(Loss):
    def __init__(self):
        super(MyBinaryCELoss, self).__init__()

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)

        # Clip predictions to prevent log(0) error
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)

        # Compute binary cross-entropy
        bce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        return tf.reduce_mean(bce)

### Custom Layer

In [23]:
from tensorflow.keras.layers import Layer

class MyDense(Layer):
    def __init__(self, units=32, activation=None):
        super(MyDense, self).__init__()
        self.units = units
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        w_init = tf.random_normal_initializer()
        self.w = tf.Variable(name="kernel",
            initial_value=w_init(shape=(input_shape[-1], self.units),
                                 dtype='float32'),
            trainable=True)
        b_init = tf.zeros_initializer()
        self.b = tf.Variable(name="bias",
            initial_value=b_init(shape=(self.units,), dtype='float32'),
            trainable=True)
        super().build(input_shape)

    def call(self, inputs):
        return self.activation(tf.matmul(inputs, self.w) + self.b)

### Custom Model

In [24]:
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K

class MyBlock(Model):
    def __init__(self, units):
        super(MyBlock, self).__init__()
        self.seq = Sequential([
            layers.Conv2D(units, 3, activation='relu'),
            layers.MaxPooling2D(),
        ])

    def call(self, inputs):
        return self.seq(inputs)

class MyModel(Model):
    def __init__(self):
        super(MyModel, self).__init__()

        self.block1 = MyBlock(32)
        self.block2 = MyBlock(64)
        self.flatten = layers.Flatten()
        self.dense1 = MyDense(64, activation='relu')
        self.dense2 = MyDense(32, activation=None)

        # Lambda Custom Layer
        self.MyLambdaLayer = layers.Lambda(lambda x: K.maximum(-0.1, x))
        self.dense3 = MyDense(1, activation='sigmoid')

    def call(self, inputs):
        x = self.block1(inputs)
        x = self.block2(x)
        x = self.flatten(x)
        x = self.dense1(x)
        x = self.dense2(x)
        x = self.MyLambdaLayer(x)
        return self.dense3(x)

### Model Training

In [25]:
epochs = 5
learning_rate = 1e-4
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
loss_fn = MyBinaryCELoss()

model = MyModel()
model.compile(optimizer=optimizer, loss=loss_fn, metrics=["accuracy"])

In [26]:
model.fit(x=train_ds, validation_data=val_ds, epochs=epochs)

Epoch 1/5
21/21 [==============================] - 6s 165ms/step - loss: 1.3619 - accuracy: 0.6433 - val_loss: 0.2412 - val_accuracy: 0.8875
Epoch 2/5
21/21 [==============================] - 3s 149ms/step - loss: 0.1982 - accuracy: 0.9190 - val_loss: 0.1331 - val_accuracy: 0.9563
Epoch 3/5
21/21 [==============================] - 5s 223ms/step - loss: 0.1014 - accuracy: 0.9595 - val_loss: 0.1326 - val_accuracy: 0.9563
Epoch 4/5
21/21 [==============================] - 3s 149ms/step - loss: 0.0725 - accuracy: 0.9798 - val_loss: 0.0889 - val_accuracy: 0.9750
Epoch 5/5
21/21 [==============================] - 3s 150ms/step - loss: 0.0668 - accuracy: 0.9798 - val_loss: 0.1130 - val_accuracy: 0.9563
